# [4/4] TỔNG HỢP – Báo cáo so sánh CNN vs RNN vs Transformer
### Chạy CUỐI CÙNG, sau khi 3 notebook nhánh đã Commit xong. Không cần GPU (Accelerator: None).

**Cách nạp kết quả:** panel bên phải → **+ Add Input → Your Work** → add output của CẢ 3 notebook (01_CNN, 02_RNN, 03_Transformer). Notebook tự tìm mọi file `results/summary.csv` trong `/kaggle/input/`.

In [ ]:
import glob, os, json
import pandas as pd
import matplotlib.pyplot as plt

# ===== 0. KIỂM TRA version output của từng notebook input có phải bản FULL RUN không =====
# Quick-test/cancelled: mọi stage *_final có train_min ~1 phút -> từ chối, yêu cầu pin lại version.
state_files = sorted(glob.glob("/kaggle/input/**/state.json", recursive=True))
print("Tìm thấy", len(state_files), "state.json")
bad = []
for sf in state_files:
    slug = os.path.basename(os.path.dirname(sf))
    st = json.load(open(sf))
    fins = [v for k, v in st.items() if k.endswith("_final") and isinstance(v, dict)]
    maxmin = max((float(v.get("train_min", 0)) for v in fins), default=0)
    n_stage = len(st)
    # Notebook detection (01/03) train final hàng chục-trăm phút -> ngưỡng 3'.
    # Notebook RNN (02) rất nhẹ: full ~2'/model, quick-test ~0.1-0.5' -> ngưỡng 1'.
    has_det = any(k in st for k in ("yolo_final", "frcnn_final", "rtdetr_final"))
    thr = 3.0 if has_det else 1.0
    status = "OK (full run)" if maxmin >= thr else "❌ QUICK-TEST/RÁC"
    print(f"  {slug}: {n_stage} stage, max train_min={maxmin:.1f} (ngưỡng {thr}) -> {status}")
    if maxmin < thr: bad.append(slug)
assert not bad, f"Input các notebook sau đang trỏ version RÁC (quick-test): {bad}. Vào panel Input -> chọn/pin đúng version full run rồi chạy lại!"
assert len(state_files) >= 3, "Cần Add Input output của đủ 3 notebook nhánh (01, 02, 03)!"

# ===== 1. Nạp kết quả chuẩn hóa =====
paths = sorted(glob.glob("/kaggle/input/**/results/summary.csv", recursive=True))
print("summary.csv:", paths)
assert len(paths) >= 3, "Thiếu results/summary.csv — kiểm tra lại version output của từng notebook!"
df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
display(df)

## 1. Bảng so sánh DETECTION (CNN vs Transformer) trên TEST

In [ ]:
df_det = df[df.task=="detection"].set_index("model").sort_values("mAP50_test", ascending=False)
cols = [c for c in ["branch","cfg","mAP50_val","mAP50_test","mAP50_95_test","precision_test","recall_test","infer_ms","train_min"] if c in df_det.columns]
df_det = df_det[cols]
df_det.to_csv("/kaggle/working/final_detection_compare.csv")
display(df_det)

## 2. Bảng so sánh CLASSIFICATION 12 lớp (CNN vs RNN vs Transformer) trên TEST

In [ ]:
df_cls = df[df.task=="classification"].set_index("model").sort_values("macro_f1", ascending=False)
cols = [c for c in ["branch","cfg","accuracy","macro_f1","worst_class_f1","train_min"] if c in df_cls.columns]
df_cls = df_cls[cols]
df_cls.to_csv("/kaggle/working/final_classification_compare.csv")
display(df_cls)

## 3. Biểu đồ + chọn model tốt nhất

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,4.5))
df_det[["mAP50_test"]].astype(float).plot.bar(ax=axes[0], rot=15, legend=False, title="Detection – mAP@0.5 (TEST)")
df_cls[["macro_f1","worst_class_f1"]].astype(float).plot.bar(ax=axes[1], rot=15, title="Classification – macro-F1 & F1 lớp yếu nhất (TEST)")
plt.tight_layout(); plt.savefig("/kaggle/working/final_compare.png", dpi=150); plt.show()

best_det = df_det["mAP50_test"].astype(float).idxmax()
best_cls = df_cls["macro_f1"].astype(float).idxmax()
best_branch_cls = df_cls.loc[best_cls, "branch"]
print(f">>> Detector tốt nhất: {best_det}")
print(f">>> Classifier tốt nhất: {best_cls} (nhánh {best_branch_cls})")
print(f">>> Pipeline Active AI Camera đề xuất: {best_det} + temporal smoothing bằng RNN trên chuỗi frame.")

## 4. So sánh với Top 3 baseline GitHub (2023–2026)

In [ ]:
baseline_rows = [
    {"Nguồn": f"Đồ án này – {best_det}", "Dataset": "PPE-Unified v1, 12 lớp (test 10%)",
     "mAP50": float(df_det.loc[best_det, "mAP50_test"]), "Ghi chú": "HPO đầy đủ, split cố định seed 42, tái lập được"},
    {"Nguồn": "snehilsanyal/Construction-Site-Safety-PPE-Detection (YOLOv8n)", "Dataset": "Roboflow CSS 2.801 ảnh, 10 lớp",
     "mAP50": None, "Ghi chú": "KHÔNG công bố số liệu; chỉ có curve/confusion matrix"},
    {"Nguồn": "jomarkow/Safety-Helmet-Detection (YOLOv8n)", "Dataset": "Kaggle Hard Hat 5k (khác dataset)",
     "mAP50": None, "Ghi chú": "KHÔNG công bố số liệu; 10 epochs proof-of-concept"},
    {"Nguồn": "lyuwenyu/RT-DETR (R18 official)", "Dataset": "COCO val2017 (80 lớp)",
     "mAP50": None, "Ghi chú": "official 46.5 mAP50-95 trên COCO – chỉ tham chiếu, khác dataset"},
]
df_base = pd.DataFrame(baseline_rows)
df_base.to_csv("/kaggle/working/baseline_compare.csv", index=False)
display(df_base)

## 5. Sinh báo cáo markdown + README.md cho GitHub

In [ ]:
hpo_files = sorted(glob.glob("/kaggle/input/**/results/hpo_*.csv", recursive=True))
hpo_md = ""
for p in hpo_files:
    name = os.path.basename(p).replace("hpo_","").replace(".csv","")
    hpo_md += f"\n### HPO – {name}\n\n" + pd.read_csv(p).to_markdown(index=False) + "\n"

report = f"""# Báo cáo so sánh: CNN vs RNN vs Transformer – Safety-PPE Detection 12 lớp
Đồ án Deep Learning · Active AI Camera – An toàn lao động nhà máy · Lâm Quốc Đạt

## 1. Dataset & protocol
- PPE-Unified v1 (gộp 3 nguồn CC BY 4.0, khử leakage theo ảnh gốc + MD5, chuẩn hóa 640×640), 12 lớp = 6 cặp tuân thủ/vi phạm, 17.951 ảnh.
- Split cố định 70/20/10 theo nhóm ảnh gốc (group-aware, seed 42), dùng chung cho cả 3 nhánh; TEST chỉ dùng 1 lần. Public: kaggle.com/datasets/quocdatlam/ppe-unified-v1
- Chống mất cân bằng: repeat-factor oversampling (train), undersampling crop, class weights, chọn model theo macro-F1.

## 2. Kết quả DETECTION (TEST)
{df_det.to_markdown()}

## 3. Kết quả CLASSIFICATION 12 lớp (TEST)
{df_cls.to_markdown()}

## 4. Model được chọn
- Detection: **{best_det}** | Classification: **{best_cls}**
- Pipeline triển khai: {best_det} + RNN temporal smoothing trên chuỗi frame camera.

## 5. So sánh baseline GitHub 2023–2026
{df_base.to_markdown(index=False)}

## 6. Phụ lục: các bảng sàng lọc hyperparameter
{hpo_md}
"""
with open("/kaggle/working/BaoCao_SoSanh.md","w",encoding="utf-8") as f: f.write(report)

readme = report.replace("# Báo cáo so sánh", "# Helmet/PPE Detection – CNN vs RNN vs Transformer\n\n> Full comparison report")
with open("/kaggle/working/README.md","w",encoding="utf-8") as f: f.write(readme)
print("Đã xuất: BaoCao_SoSanh.md, README.md, final_*.csv, final_compare.png")
print("Cấu trúc repo GitHub gợi ý: README.md + 4 notebook + results/ (csv, png) + weights qua GitHub Release")

## 6. Gom TOÀN BỘ output của 3 notebook nhánh vào output notebook này
Copy mọi thứ (state.json, results/, runs/ có weights...) TRỪ thư mục `dataset/` và file pretrained `.pt` gốc — để chỉ cần Download output notebook 04 là có đủ kết quả cả đồ án, không kèm 2GB dataset.

In [ ]:
import shutil

EXCLUDE_DIRS = {"dataset", "__pycache__"}
collected = 0
for state in sorted(glob.glob("/kaggle/input/**/state.json", recursive=True)):
    src_root = os.path.dirname(state)
    slug = os.path.basename(src_root)
    dst_root = f"/kaggle/working/collected/{slug}"
    os.makedirs(dst_root, exist_ok=True)
    for item in os.listdir(src_root):
        if item in EXCLUDE_DIRS: continue
        if item.endswith(".pt"): continue   # yolo26s.pt/rtdetr-l.pt pretrained gốc, không phải kết quả
        s_, d_ = os.path.join(src_root, item), os.path.join(dst_root, item)
        if os.path.isdir(s_): shutil.copytree(s_, d_, dirs_exist_ok=True)
        else: shutil.copy2(s_, d_)
    collected += 1
    print("Đã gom:", slug, "->", dst_root)
print("Tổng notebook gom được:", collected, "(cần 3: notebook 01, 02, 03)")
total = 0
for r, _, files in os.walk("/kaggle/working/collected"):
    total += sum(os.path.getsize(os.path.join(r, f)) for f in files)
print(f"Dung lượng collected/: {total/1e9:.2f} GB")